In [3]:
import sympy as sp

# 定义符号变量
z, y = sp.symbols('z y', real=True)
mu_z, mu_y, b = sp.symbols('mu_z mu_y b', real=True)
Sigma_z, Sigma_y = sp.symbols('Sigma_z Sigma_y', real=True, positive=True)
W = sp.MatrixSymbol('W', sp.Symbol('n_z'), sp.Symbol('n_y'))  # 权重矩阵，维度 n_z × n_y

# 定义逆矩阵（符号表示）
Sigma_z_inv = sp.MatrixSymbol('Sigma_z_inv', sp.Symbol('n_z'), sp.Symbol('n_z'))
Sigma_y_inv = sp.MatrixSymbol('Sigma_y_inv', sp.Symbol('n_y'), sp.Symbol('n_y'))

# 定义条件协方差矩阵的逆
Sigma_z_given_y_inv = Sigma_z_inv + W.T * Sigma_y_inv * W

# 定义条件均值
term1 = W.T * Sigma_y_inv * (y - b)
term2 = Sigma_z_inv * mu_z
mu_z_given_y = Sigma_z_given_y_inv.inv() * (term1 + term2)

# 输出结果
Sigma_z_given_y_inv


Sigma_z_inv + W.T*Sigma_y_inv*W

In [4]:
 mu_z_given_y

(Sigma_z_inv + W.T*Sigma_y_inv*W)**(-1)*(mu_z*Sigma_z_inv + (-b + y)*W.T*Sigma_y_inv)

In [3]:
import sympy as sp
from sympy.stats import MultivariateNormal, density, Normal, marginal_distribution

# ========== 1. 定义符号变量 ==========
mu1, mu2 = sp.symbols('mu1 mu2', real=True)
sigma1, sigma2 = sp.symbols('sigma1 sigma2', positive=True)
rho = sp.symbols('rho', real=True)

# 协方差矩阵 Sigma
# 根据二元正态分布，协方差矩阵为：
# Sigma = [[sigma1^2, rho*sigma1*sigma2],
#          [rho*sigma1*sigma2, sigma2^2]]
Sigma = sp.Matrix([[sigma1**2, rho * sigma1 * sigma2],
                   [rho * sigma1 * sigma2, sigma2**2]])

# 均值向量
mu = sp.Matrix([mu1, mu2])

# ========== 2. 定义二元正态分布 ==========
X = MultivariateNormal('X', mu, Sigma)

# 查看联合密度
x1, x2 = sp.symbols('x1 x2', real=True)
joint_density = density(X)(x1, x2)
print("=== 联合密度函数 ===")
sp.pprint(sp.simplify(joint_density))

# ========== 3. 推导条件分布 ==========
# 方法一：使用 SymPy 内置的 marginal_distribution 和 conditional_distribution
# 注意：SymPy 的 marginal_distribution 返回的是边缘分布（一个一元函数）

# 获取 X2 的边缘分布（用于条件分布分母）
marginal_X2 = marginal_distribution(X, X[1])  # X[1] 是第二个分量，即 X2
print("\n=== X2 的边缘密度 ===")
sp.pprint(marginal_X2(x2))

# 条件分布：p(X1 | X2 = x2)
# SymPy 中的 conditional_distribution 可以直接给出条件分布
from sympy.stats import ConditionalDistribution

# 直接使用条件分布定义
cond_dist = ConditionalDistribution(X[0], X[1], x2)

# 但对于二元正态，我们可以精确推导
# 条件均值：mu1 + rho * (sigma1/sigma2) * (x2 - mu2)
# 条件方差：sigma1^2 * (1 - rho^2)

cond_mean = mu1 + rho * (sigma1 / sigma2) * (x2 - mu2)
cond_var = sigma1**2 * (1 - rho**2)

print("\n=== 条件分布 X1 | X2 = x2 ===")
print("条件均值 E[X1 | X2] =", cond_mean)
print("条件方差 Var(X1 | X2) =", cond_var)

# 条件密度函数（手动构建）
cond_density = (1 / sp.sqrt(2 * sp.pi * cond_var)) * \
               sp.exp(-((x1 - cond_mean)**2) / (2 * cond_var))
print("\n条件密度函数 p(x1 | x2) =")
sp.pprint(cond_density)

# ========== 4. 验证条件密度积分等于 1 ==========
# 对 x1 积分，应该等于 1
integral_check = sp.integrate(cond_density, (x1, -sp.oo, sp.oo))
print("\n验证条件密度积分（应等于1）:", sp.simplify(integral_check))

# ========== 5. 验证联合密度 = 边缘密度 * 条件密度 ==========
# 联合密度 = 边缘密度(X2) * 条件密度(X1|X2)
factorized_density = sp.simplify(marginal_X2(x2) * cond_density)
print("\n联合密度分解验证（是否与原始联合密度一致）:")
print("差异为0表示一致：", sp.simplify(factorized_density - joint_density))

=== 联合密度函数 ===
   2   2                                              2        2   2           ↪
 μ₁ ⋅σ₂  - 2⋅μ₁⋅μ₂⋅ρ⋅σ₁⋅σ₂ + 2⋅μ₁⋅ρ⋅σ₁⋅σ₂⋅x₂ - 2⋅μ₁⋅σ₂ ⋅x₁ + μ₂ ⋅σ₁  + 2⋅μ₂⋅ρ⋅ ↪
 ───────────────────────────────────────────────────────────────────────────── ↪
                                                             2   2             ↪
                                                         2⋅σ₁ ⋅σ₂ ⋅(ρ - 1)⋅(ρ  ↪
ℯ                                                                              ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                       _______ ↪
                                                                      ╱      2 ↪
                                                          2⋅π⋅σ₁⋅σ₂⋅╲╱  1 - ρ  ↪

↪                   2                          2   2     2   2
↪ σ₁⋅σ₂⋅x₁ - 2⋅μ₂⋅σ₁ ⋅x₂ - 2⋅ρ⋅σ₁⋅σ₂⋅x₁⋅x₂ + σ₁ ⋅x₂  + σ₂ ⋅x₁ 
↪ ──────────────────────────────────────────────

ImportError: cannot import name 'ConditionalDistribution' from 'sympy.stats' (/Users/lunarcheung/miniconda3/lib/python3.14/site-packages/sympy/stats/__init__.py)